In [1]:
import os

import pandas as pd
import numpy as np
from PIL import Image
from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score

from transformers import AutoImageProcessor, AutoModel

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CXR_JPG_ROOT = '/home/DAHS1/mimic-cxr-jpg-2.0.0'

In [2]:
final_processed_cxr_df = pd.read_feather('/home/DAHS1/gangmin/my_research/clinical_multimodal_learning/data/full_data/subject_data/final_cxr_df_20260713.ftr')

In [3]:
final_processed_cxr_df.columns

Index(['subject_id', 'study_id', 'dicom_id', 'image_path', 'ViewPosition',
       'cxrtime', 'lung_mask_path', 'label_cardiomegaly', 'mask_cardiomegaly',
       'loc_cardiomegaly', 'label_pneumonia', 'mask_pneumonia',
       'loc_pneumonia', 'label_atelectasis', 'mask_atelectasis',
       'loc_atelectasis', 'label_opacity', 'mask_opacity', 'loc_opacity',
       'label_consolidation', 'mask_consolidation', 'loc_consolidation',
       'label_edema', 'mask_edema', 'loc_edema', 'label_effusion',
       'mask_effusion', 'loc_effusion', 'report', 'txt_path'],
      dtype='object')

In [ ]:
# label_cols = [
#     'label_cardiomegaly', 'label_pneumonia', 'label_atelectasis', 'label_opacity', 'label_consolidation', 'label_edema', 'label_effusion'
# ]


label_cols = [
    'label_edema', 'label_cardiomegaly', 'label_pneumonia', 'label_effusion'
]

In [5]:
label_df = final_processed_cxr_df[label_cols]
label_df.isnull().sum()

label_cardiomegaly     51761
label_pneumonia        51761
label_atelectasis      51761
label_opacity          51761
label_consolidation    51761
label_edema            51761
label_effusion         51761
dtype: int64

In [ ]:
label_df[label_df['label_edema']==1]

,label_cardiomegaly,label_pneumonia,label_atelectasis,label_opacity,label_consolidation,label_edema,label_effusion
109,0.0,0.0,0.0,1.0,0.0,0.0,1.0
128,0.0,0.0,0.0,1.0,0.0,0.0,0.0
157,0.0,0.0,0.0,1.0,0.0,0.0,1.0
170,1.0,0.0,0.0,1.0,0.0,0.0,1.0
185,0.0,0.0,0.0,1.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...
243230,1.0,0.0,1.0,1.0,0.0,1.0,0.0
243248,1.0,0.0,1.0,1.0,0.0,0.0,0.0
243250,0.0,0.0,0.0,1.0,0.0,1.0,0.0
243309,0.0,0.0,1.0,1.0,1.0,0.0,0.0


In [7]:
cxr_df = final_processed_cxr_df[final_processed_cxr_df[label_cols].notna().any(axis=1)]
cxr_df = cxr_df.drop_duplicates(subset=['dicom_id']).reset_index(drop=True)

print(f"Unique dicom_id: {len(cxr_df)}   Unique subject_id: {cxr_df['subject_id'].nunique()}")

subject_ids = cxr_df['subject_id'].unique()

train_ids, temp_ids = train_test_split(subject_ids, test_size=0.30, random_state=42)
val_ids, test_ids = train_test_split(temp_ids, test_size=0.50, random_state=42)

train_df = cxr_df[cxr_df['subject_id'].isin(train_ids)].reset_index(drop=True)
val_df   = cxr_df[cxr_df['subject_id'].isin(val_ids)].reset_index(drop=True)
test_df  = cxr_df[cxr_df['subject_id'].isin(test_ids)].reset_index(drop=True)

print(f"Train / Val / Test: {len(train_df)} / {len(val_df)} / {len(test_df)}")

# label prevalence (ignores NaN)
prev = train_df[label_cols].apply(lambda s: s.mean(skipna=True))
print("\nTrain positive prevalence per label:")
print(prev.round(4))

Unique dicom_id: 191563   Unique subject_id: 61580
Train / Val / Test: 133802 / 28801 / 28960

Train positive prevalence per label:
label_cardiomegaly     0.2132
label_pneumonia        0.0237
label_atelectasis      0.0460
label_opacity          0.0450
label_consolidation    0.0170
label_edema            0.0757
label_effusion         0.0524
dtype: float64


In [8]:
class CXRDataset(Dataset):
    def __init__(self, df, processor, label_cols, image_root=CXR_JPG_ROOT):
        self.df = df.reset_index(drop=True)
        self.processor = processor
        self.label_cols = label_cols
        self.image_root = image_root

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image_path = os.path.join(self.image_root, row['image_path'])
        image = Image.open(image_path).convert("RGB")

        processed = self.processor(images=image, return_tensors="pt")

        pixel_values = processed['pixel_values'].squeeze(0)

        labels_raw = row[self.label_cols].to_numpy(dtype=np.float32)
        label_mask = ~np.isnan(labels_raw)
        labels = np.nan_to_num(labels_raw, nan=0.0)

        return {
            "pixel_values": pixel_values,
            "labels": torch.from_numpy(labels),
            "label_mask": torch.from_numpy(label_mask),
            "image_path": image_path
        }


class RadDinoClassifier(nn.Module):
    def __init__(self, num_classes: int, freeze_encoder: bool=True, dropout: float=0.1):
        super().__init__()
        
        self.encoder = AutoModel.from_pretrained("microsoft/rad-dino")
        hidden_size = self.encoder.config.hidden_size

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size, num_classes)
        )

        self.freeze_encoder = freeze_encoder
        if freeze_encoder:
            for param in self.encoder.parameters():
                param.requires_grad = False
        
    def forward(self, pixel_values):
        if self.freeze_encoder:
            with torch.no_grad():
                outputs = self.encoder(pixel_values=pixel_values)
        else:
            outputs = self.encoder(pixel_values=pixel_values)

        cls_embedding = outputs.last_hidden_state[:, 0, :]
        logits = self.classifier(cls_embedding)

        return logits
    

def masked_bce_with_logits_loss(logits, labels, label_mask):
    criterion = nn.BCEWithLogitsLoss(reduction='none')
    loss = criterion(logits, labels)

    label_mask_f = label_mask.float()
    valid_count = label_mask_f.sum()

    if valid_count == 0:
        return logits.sum() * 0.0

    masked_loss = loss * label_mask_f
    return masked_loss.sum() / valid_count

In [9]:
processor = AutoImageProcessor.from_pretrained("microsoft/rad-dino")

model = RadDinoClassifier(num_classes=7, freeze_encoder=True)

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

In [10]:
BATCH_SIZE = 128
NUM_WORKERS = 8
LR = 1e-4
WEIGHT_DECAY = 1e-4
EPOCHS = 30

train_ds = CXRDataset(train_df, processor, label_cols)
val_ds   = CXRDataset(val_df,   processor, label_cols)
test_ds  = CXRDataset(test_df,  processor, label_cols)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True, drop_last=True,
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
)
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
)

model = model.to(device)

# encoder frozen -> only classifier head params are trained
trainable_params = [p for p in model.parameters() if p.requires_grad]
n_trainable = sum(p.numel() for p in trainable_params)
print(f"Trainable params: {n_trainable:,}")

optimizer = AdamW(trainable_params, lr=LR, weight_decay=WEIGHT_DECAY)

Trainable params: 5,383


In [11]:
@torch.no_grad()
def evaluate(model, loader, label_cols, device, desc="eval"):
    model.eval()
    all_probs, all_labels, all_masks = [], [], []
    total_loss, total_valid = 0.0, 0.0

    for batch in tqdm(loader, desc=desc, leave=False):
        pixel_values = batch['pixel_values'].to(device, non_blocking=True)
        labels = batch['labels'].to(device, non_blocking=True)
        label_mask = batch['label_mask'].to(device, non_blocking=True)

        logits = model(pixel_values)
        loss = masked_bce_with_logits_loss(logits, labels, label_mask)

        valid = label_mask.float().sum().item()
        total_loss += loss.item() * valid
        total_valid += valid

        all_probs.append(torch.sigmoid(logits).cpu().numpy())
        all_labels.append(labels.cpu().numpy())
        all_masks.append(label_mask.cpu().numpy())

    probs  = np.concatenate(all_probs, axis=0)
    labels = np.concatenate(all_labels, axis=0)
    masks  = np.concatenate(all_masks, axis=0).astype(bool)

    per_label = {}
    aurocs, auprcs = [], []
    for i, name in enumerate(label_cols):
        m = masks[:, i]
        y = labels[m, i]
        p = probs[m, i]
        if m.sum() < 2 or len(np.unique(y)) < 2:
            per_label[name] = {'auroc': float('nan'), 'auprc': float('nan'),
                               'n': int(m.sum()), 'pos': int(y.sum())}
            continue
        auroc = roc_auc_score(y, p)
        auprc = average_precision_score(y, p)
        per_label[name] = {'auroc': auroc, 'auprc': auprc,
                           'n': int(m.sum()), 'pos': int(y.sum())}
        aurocs.append(auroc)
        auprcs.append(auprc)

    macro_auroc = float(np.mean(aurocs)) if aurocs else float('nan')
    macro_auprc = float(np.mean(auprcs)) if auprcs else float('nan')
    avg_loss = total_loss / max(total_valid, 1.0)

    return {
        'loss': avg_loss,
        'macro_auroc': macro_auroc,
        'macro_auprc': macro_auprc,
        'per_label': per_label,
    }


def print_metrics(metrics, title):
    print(f"\n[{title}]  loss={metrics['loss']:.4f}   "
          f"macro-AUROC={metrics['macro_auroc']:.4f}   "
          f"macro-AUPRC={metrics['macro_auprc']:.4f}")
    print(f"  {'label':<24}{'n':>8}{'pos':>8}{'AUROC':>10}{'AUPRC':>10}")
    for name, s in metrics['per_label'].items():
        print(f"  {name:<24}{s['n']:>8}{s['pos']:>8}"
              f"{s['auroc']:>10.4f}{s['auprc']:>10.4f}")

In [12]:
best_val_auroc = -np.inf
best_state = None

for epoch in range(1, EPOCHS + 1):
    model.train()
    if model.freeze_encoder:
        model.encoder.eval()

    running_loss, running_valid = 0.0, 0.0
    pbar = tqdm(train_loader, desc=f"epoch {epoch}/{EPOCHS}")
    for batch in pbar:
        pixel_values = batch['pixel_values'].to(device, non_blocking=True)
        labels = batch['labels'].to(device, non_blocking=True)
        label_mask = batch['label_mask'].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        logits = model(pixel_values)
        loss = masked_bce_with_logits_loss(logits, labels, label_mask)
        loss.backward()
        optimizer.step()

        valid = label_mask.float().sum().item()
        running_loss  += loss.item() * valid
        running_valid += valid
        pbar.set_postfix(loss=running_loss / max(running_valid, 1.0))

    train_loss = running_loss / max(running_valid, 1.0)
    val_metrics = evaluate(model, val_loader, label_cols, device, desc=f"val {epoch}")
    print(f"\nEpoch {epoch}: train_loss={train_loss:.4f}")
    print_metrics(val_metrics, f"val (epoch {epoch})")

    if val_metrics['macro_auroc'] > best_val_auroc:
        best_val_auroc = val_metrics['macro_auroc']
        best_state = {k: v.detach().cpu().clone() for k, v in model.classifier.state_dict().items()}
        print(f"  ** new best macro-AUROC: {best_val_auroc:.4f}")

epoch 1/30:  12%|█▏        | 127/1045 [13:30<1:37:41,  6.39s/it, loss=0.583]


KeyboardInterrupt: 

In [ ]:
# restore best classifier head and run per-label evaluation on the test set
if best_state is not None:
    model.classifier.load_state_dict(best_state)

test_metrics = evaluate(model, test_loader, label_cols, device, desc="test")
print_metrics(test_metrics, "test (best val checkpoint)")

test_per_label_df = pd.DataFrame(test_metrics['per_label']).T
test_per_label_df


[test (best val checkpoint)]  loss=0.1661   macro-AUROC=0.8678   macro-AUPRC=0.2847
  label                          n     pos     AUROC     AUPRC
  label_cardiomegaly         28960    6120    0.8537    0.6018
  label_pneumonia            28960     704    0.8513    0.1162
  label_atelectasis          28960    1290    0.8585    0.2044
  label_opacity              28960    1342    0.8720    0.2452
  label_consolidation        28960     538    0.8354    0.0836
  label_edema                28960    2226    0.9016    0.4016
  label_effusion             28960    1598    0.9020    0.3399


,auroc,auprc,n,pos
label_cardiomegaly,0.853732,0.601818,28960.0,6120.0
label_pneumonia,0.851314,0.116185,28960.0,704.0
label_atelectasis,0.858458,0.204390,28960.0,1290.0
label_opacity,0.872029,0.245245,28960.0,1342.0
label_consolidation,0.835354,0.083618,28960.0,538.0
label_edema,0.901580,0.401631,28960.0,2226.0
label_effusion,0.902028,0.339881,28960.0,1598.0


In [ ]:
import json
from datetime import datetime
from pathlib import Path

save_dir = Path('/home/DAHS1/gangmin/my_research/clinical_multimodal_learning/checkpoints/cxr_linear_head')
save_dir.mkdir(parents=True, exist_ok=True)

tag = datetime.now().strftime('%Y%m%d_%H%M%S')
ckpt_path = save_dir / f'raddino_linear_head_{tag}.pt'
meta_path = save_dir / f'raddino_linear_head_{tag}.json'

torch.save({
    'classifier_state_dict': best_state,
    'label_cols': label_cols,
    'encoder_name': 'microsoft/rad-dino',
    'hidden_size': model.encoder.config.hidden_size,
    'num_classes': len(label_cols),
    'freeze_encoder': True,
    'best_val_macro_auroc': best_val_auroc,
    'test_metrics': {
        'macro_auroc': test_metrics['macro_auroc'],
        'macro_auprc': test_metrics['macro_auprc'],
        'per_label': test_metrics['per_label'],
    },
}, ckpt_path)

with open(meta_path, 'w') as f:
    json.dump({
        'ckpt': str(ckpt_path),
        'label_cols': label_cols,
        'best_val_macro_auroc': best_val_auroc,
        'test_macro_auroc': test_metrics['macro_auroc'],
        'test_macro_auprc': test_metrics['macro_auprc'],
    }, f, indent=2)

print(f'saved head -> {ckpt_path}')
print(f'saved meta -> {meta_path}')

# ckpt = torch.load(ckpt_path, map_location='cpu')
# assert ckpt['label_cols'] == label_cols_multimodal, "라벨 순서 불일치"
# model.classifier.load_state_dict(ckpt['classifier_state_dict'])

saved head -> /home/DAHS1/gangmin/my_research/clinical_multimodal_learning/checkpoints/cxr_linear_head/raddino_linear_head_20260715_053311.pt
saved meta -> /home/DAHS1/gangmin/my_research/clinical_multimodal_learning/checkpoints/cxr_linear_head/raddino_linear_head_20260715_053311.json
